# E1 · W9 · Kaggle run 19 — FBD-SV-2024: chuẩn hóa + đo cảnh báo sai trên neg_eval (GPU T4)
**Mục đích (W9-03 #131; `docs/w9/W9_PLAN.md` mục D; ADR-0013, ADR-0014):** kiểm SHA-256 483 clip, chuẩn hóa (cắt giữa 5:4, 640×512, 30 fps), chạy `e1pipe` cấu hình C0, đếm cảnh báo sai R0/R1 và frame có detection Drone. Mọi cảnh báo trên clip âm tính là cảnh báo sai.

**ADR-0014:** dataset `e1-fbd-sv-2024` phải ở chế độ **Private**; video chuẩn hóa chỉ nằm trong `/tmp` của phiên và bị xóa; bằng chứng chỉ có số tổng hợp và ID clip. **Không** tải video/frame ra khỏi Kaggle.


In [ ]:
COMMIT = ""                      # <-- SHA day du 40 ky tu (cung commit voi dataset e1-w9-tools)
WEIGHTS_SHA = "a3de1e0195e4380aac497612296ed57f67e41ceb5ae3314e82a5ad66b125ef8a"
ULTRALYTICS_VERSION = "8.4.183"
RUNNER = "manual"


In [ ]:
import glob, hashlib, json, os, subprocess, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu (xem huong dan LOCAL)"
if not glob.glob("/kaggle/input/**/TOOLS_MANIFEST.json", recursive=True):
    zipfile.ZipFile(find("e1-w9-tools.zip")).extractall("/kaggle/working/tools")
TOOLS = str(Path((glob.glob("/kaggle/input/**/TOOLS_MANIFEST.json", recursive=True) or glob.glob("/kaggle/working/tools/**/TOOLS_MANIFEST.json", recursive=True))[0]).parent)
tm = json.load(open(f"{TOOLS}/TOOLS_MANIFEST.json"))
assert tm["commit"] == COMMIT, f"dataset tools dung commit {tm['commit'][:12]}, khac COMMIT — tao lai tools bang make_tools_bundle.py"
assert not tm["dirty_worktree"], "tools dung tu thu muc co thay doi chua commit"
bad = [f["path"] for f in tm["files"] if sha(f"{TOOLS}/{f['path']}") != f["sha256"]]
assert not bad, f"tep tools lech SHA: {bad[:5]}"
W = [p for p in sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True)) if sha(p) == WEIGHTS_SHA]
assert W, "Khong thay best.pt co SHA-256 cua yolo11n_640_s20261030 — kiem tra input e1-w5-run13"
WEIGHTS = W[0]
print("TOOLS", TOOLS, "| commit", tm["commit"][:12], "|", len(tm["files"]), "tep | WEIGHTS", WEIGHTS)


In [ ]:
!pip -q install "lap>=0.5.12" {"ultralytics==" + ULTRALYTICS_VERSION}
import lap, torch, ultralytics
print("ultralytics", ultralytics.__version__, "| lap", lap.__version__, "| cuda", torch.cuda.is_available())
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU T4 x2"


In [ ]:
FBD = sorted(glob.glob("/kaggle/input/**/videos/train/bird_1.mp4", recursive=True))
assert FBD, "Khong thay FBD-SV-2024 (videos/train/bird_1.mp4) — kiem tra input e1-fbd-sv-2024"
FBD_ROOT = str(Path(FBD[0]).parents[3])
n = len(glob.glob(f"{FBD_ROOT}/**/videos/*/*.mp4", recursive=True)); print("FBD_ROOT", FBD_ROOT, "| mp4:", n)
assert n == 483, "So mp4 khac 483"
NEG = f"{TOOLS}/scripts/kaggle/w9_neg_fbdsv.py"


In [ ]:
def run_logged(cmd):
    """Chay job con, chuyen tung dong log ra o notebook ngay lap tuc (Kaggle Log thay tien do)."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

BASE = [sys.executable, NEG, "--fbd-root", FBD_ROOT, "--tools", TOOLS, "--weights", WEIGHTS, "--weights-sha", WEIGHTS_SHA,
        "--commit", COMMIT, "--runner", RUNNER]
# Chay thu 10 clip (khoang 1-2 phut) de thay toc do truoc khi chay du; ket qua chay thu nam o /tmp, khong vao Output.
rc_smoke = run_logged(BASE + ["--max-clips", "10", "--tag", "smoke", "--out", "/tmp/e1_smoke", "--work", "/tmp/e1_fbd_norm_smoke"])
print("chay thu: ma thoat", rc_smoke)
assert rc_smoke == 0, "Chay thu 10 clip loi: gui log o tren cho Claude, KHONG chay tiep"


In [ ]:
rc = run_logged(BASE + ["--out", "/kaggle/working"])
print("ma thoat", rc)


## Tự kiểm tra (phải in `PASS`)

In [ ]:
d = "/kaggle/working/w9_neg_fbdsv"
st = json.load(open(f"{d}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
s = json.load(open(f"{d}/neg_summary.json"))
print(json.dumps({k: s[k] for k in ("clips", "minutes", "R1", "R0", "frac_frames_drone_det", "adr0011_trigger")}, indent=1, ensure_ascii=False))
leak = [p for p in glob.glob(f"{d}/**/*", recursive=True) if p.lower().endswith((".mp4", ".jpg", ".png", ".jpeg"))]
ok = rc == 0 and all(v["status"] == "OK" for v in st.values()) and s["clips"] == 483 and not leak and not os.path.exists("/tmp/e1_fbd_norm") and not os.path.exists("/tmp/e1_fbd_norm_smoke")
print("PASS" if ok else f"FAIL - gui stage_status.json cho Claude (leak={leak[:3]})")
z = "/kaggle/working/w9_neg_fbdsv_evidence.zip"; print(z, os.path.getsize(z) // 1024, "KB  sha256", sha(z))
